# LeNet-5 CNN 板上推理 + 多频率 Bitstream 选择 + Profiler

流程与 `lenet5_test_pynq.ipynb` 基本一致，区别在于第 1 个 code cell 支持按 bitstream / 频率 profile 选择。

常用选择：
- `SELECT_PROFILE = "55mhz"`：优先使用 `cim_soc_55mhz.bit`
- `SELECT_PROFILE = "60mhz"`：使用 `cim_soc.bit`
- `SELECT_PROFILE = "auto"`：按 55MHz → 60MHz 顺序自动选已存在的 bitstream
- `SELECT_PROFILE = "custom"`：自定义 bitstream 文件名和 fabric 频率


In [1]:
# ====================================================================
# 1. 导入 + 选择 bitstream / 频率 profile + 加载 driver
# ====================================================================
import glob
import os
import time
import zipfile

import numpy as np

from cim_driver import (
    CIMDriver, CIMModel, _make_run_id,
    weight_to_chunks, bias_to_u32,
    _BASE, _MMIO_SIZE,
)
import golden_model as gm

# ---- profile 选择 ----
# 可选: "auto", "55mhz", "60mhz", "custom"
SELECT_PROFILE = "55mhz"

# 需要 legacy MMIO 时保持 False；已确认 DMA 新 bitstream 稳定后再改 True
USE_DMA = False

BITSTREAM_PROFILES = {
    "55mhz": {
        "bitstream": "cim_soc_55mhz.bit",
        "fabric_mhz": 55,
        "description": "55 MHz timing-clean build",
    },
    "60mhz": {
        "bitstream": "cim_soc.bit",
        "fabric_mhz": 60,
        "description": "60 MHz baseline / C3 build",
    },
}

CUSTOM_PROFILE = {
    "bitstream": "cim_soc.bit",
    "fabric_mhz": None,
    "description": "custom bitstream",
}

def resolve_profile(select_profile, profiles, custom_profile):
    if select_profile == "custom":
        bit = custom_profile["bitstream"]
        if not os.path.exists(bit):
            raise FileNotFoundError(f"custom bitstream not found: {bit}")
        return "custom", custom_profile

    if select_profile == "auto":
        for name in ("55mhz", "60mhz"):
            bit = profiles[name]["bitstream"]
            if os.path.exists(bit):
                return name, profiles[name]
        raise FileNotFoundError(
            "No known bitstream found. Expected one of: "
            + ", ".join(p["bitstream"] for p in profiles.values())
        )

    if select_profile not in profiles:
        raise ValueError(f"Unknown SELECT_PROFILE={select_profile!r}")

    profile = profiles[select_profile]
    bit = profile["bitstream"]
    if not os.path.exists(bit):
        raise FileNotFoundError(f"bitstream not found for profile {select_profile}: {bit}")
    return select_profile, profile

PROFILE_NAME, PROFILE = resolve_profile(SELECT_PROFILE, BITSTREAM_PROFILES, CUSTOM_PROFILE)
BITSTREAM_NAME = PROFILE["bitstream"]
FABRIC_MHZ = PROFILE["fabric_mhz"]
PROFILE_DESC = PROFILE["description"]

drv = CIMDriver(BITSTREAM_NAME, use_dma=USE_DMA)

print("=== Bitstream Profile ===")
print(f"  profile    : {PROFILE_NAME}")
print(f"  bitstream  : {BITSTREAM_NAME}")
print(f"  fabric_mhz : {FABRIC_MHZ}")
print(f"  mode       : {'DMA' if USE_DMA else 'legacy MMIO'}")
print(f"  note       : {PROFILE_DESC}")
print("Overlay loaded, driver ready.")


=== Bitstream Profile ===
  profile    : 55mhz
  bitstream  : cim_soc_55mhz.bit
  fabric_mhz : 55
  mode       : legacy MMIO
  note       : 55 MHz timing-clean build
Overlay loaded, driver ready.


In [2]:
# ====================================================================
# 2. AXI 连通性测试
# ====================================================================
from pynq import MMIO

mmio = MMIO(_BASE, _MMIO_SIZE)

print(f"=== AXI Connectivity Test ({PROFILE_NAME}) ===")
status = mmio.read(0x004)
print(f"  STATUS = 0x{status:08x}")

mmio.write(0x010, 784)
rb = mmio.read(0x010)
print(f"  IN_DIM write=784, readback={rb}  {'PASS' if rb == 784 else 'FAIL'}")
assert rb == 784, "AXI read/write failed!"

mmio.write(0x014, 120)
rb = mmio.read(0x014)
print(f"  OUT_DIM write=120, readback={rb}  {'PASS' if rb == 120 else 'FAIL'}")
assert rb == 120, "AXI read/write failed on OUT_DIM!"

print("AXI OK.")


=== AXI Connectivity Test (55mhz) ===
  STATUS = 0x00000000
  IN_DIM write=784, readback=784  PASS
  OUT_DIM write=120, readback=120  PASS
AXI OK.


In [3]:
# ====================================================================
# 3. 解压 / 准备 lenet5_data
# ====================================================================
if os.path.isdir("lenet5_data"):
    print("lenet5_data/ already exists, skip unzip.")
elif os.path.exists("lenet5_data.zip"):
    with zipfile.ZipFile("lenet5_data.zip", "r") as zf:
        zf.extractall(".")
    print("Unzipped lenet5_data.zip")
else:
    raise FileNotFoundError("Need lenet5_data/ or lenet5_data.zip in current directory")


lenet5_data/ already exists, skip unzip.


In [4]:
# ====================================================================
# 4. 从 npz 加载量化参数，构建 CIMModel
# ====================================================================
DATA_DIR = "lenet5_data"
d = np.load(f"{DATA_DIR}/lenet5_qparams.npz")

model = CIMModel(drv)

# Conv1(1->6, 5x5) -> Pool1(2x2) -> Conv2(6->16, 5x5) -> Pool2(2x2)
# -> FC3(256->120) -> FC4(120->84) -> FC5(84->10)
model.add_conv(d["conv1_weight"], d["conv1_bias"],
               zp=int(d["conv1_zp"]), mult=int(d["conv1_mult"]),
               shift=int(d["conv1_shift"]), stride=1, padding=0, relu=True)
model.add_pool(2, 2)

model.add_conv(d["conv2_weight"], d["conv2_bias"],
               zp=int(d["conv2_zp"]), mult=int(d["conv2_mult"]),
               shift=int(d["conv2_shift"]), stride=1, padding=0, relu=True)
model.add_pool(2, 2)

model.add_fc(256, 120, weight_to_chunks(d["fc3_weight"]), bias_to_u32(d["fc3_bias"]),
             zp=int(d["fc3_zp"]), mult=int(d["fc3_mult"]), shift=int(d["fc3_shift"]),
             relu=True, weight_int8=d["fc3_weight"], bias_int32=d["fc3_bias"])

model.add_fc(120, 84, weight_to_chunks(d["fc4_weight"]), bias_to_u32(d["fc4_bias"]),
             zp=int(d["fc4_zp"]), mult=int(d["fc4_mult"]), shift=int(d["fc4_shift"]),
             relu=True, weight_int8=d["fc4_weight"], bias_int32=d["fc4_bias"])

model.add_fc(84, 10, weight_to_chunks(d["fc5_weight"]), bias_to_u32(d["fc5_bias"]),
             zp=int(d["fc5_zp"]), mult=int(d["fc5_mult"]), shift=int(d["fc5_shift"]),
             relu=False, weight_int8=d["fc5_weight"], bias_int32=d["fc5_bias"])

for i, layer in enumerate(model.layers):
    if layer.get("type") == "conv":
        p = layer.get("_packed")
        k = p["k_pack"] if p else 1
        print(f"  Layer {i} ({layer['C_in']}ch {layer['K_h']}x{layer['K_w']}->{layer['C_out']}ch): "
              f"col_len={layer['col_len']}, k_pack={k}")

print(f"\nModel: {len(model.layers)} layers loaded from npz")
print(f"Active profile: {PROFILE_NAME} ({FABRIC_MHZ} MHz)")


  Layer 0 (1ch 5x5->6ch): col_len=25, k_pack=21
  Layer 2 (6ch 5x5->16ch): col_len=150, k_pack=5

Model: 7 layers loaded from npz
Active profile: 55mhz (55 MHz)


In [5]:
# ====================================================================
# 5. 单图推理 + bit-exact 验证 + profiler
# ====================================================================
def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

img_dir = f"{DATA_DIR}/test_images"
img_path = sorted(glob.glob(f"{img_dir}/img_????.hex"))[0]
img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
label = int(open(img_path.replace(".hex", "_label.txt")).read().strip())

print(f"Profile: {PROFILE_NAME} ({FABRIC_MHZ} MHz), mode={'DMA' if USE_DMA else 'legacy MMIO'}")
print(f"Test image: {os.path.basename(img_path)}, label={label}\n")

print("=== Single Image LeNet-5 (verify=True, profile=True) ===")
pred, logits, prof = model.predict(
    img_u8.reshape(1, 28, 28),
    verbose=True, verify=True, profile=True,
)

print(f"\nHW pred={pred}, label={label}, {'CORRECT' if pred == label else 'WRONG'}")
print("\n--- Profiler ---")
print(f"{'Layer':<30s} {'n_mvm':>6s} {'k_pack':>6s} {'setup':>8s} {'load_x':>8s} "
      f"{'compute':>8s} {'read':>8s} {'total':>8s}")
for lp in prof["layers"]:
    if lp["type"] == "pool":
        print(f"{lp['name']:<30s} {'--':>6s} {'--':>6s} {'--':>8s} {'--':>8s} "
              f"{'--':>8s} {'--':>8s} {lp['total_ms']:>7.1f}ms")
    else:
        print(f"{lp['name']:<30s} {lp['n_mvm']:>6d} {lp['k_pack']:>6d} "
              f"{lp.get('setup_ms', 0):>7.1f}ms {lp.get('load_x_ms', 0):>7.1f}ms "
              f"{lp.get('compute_ms', 0):>7.1f}ms {lp.get('read_out_ms', 0):>7.1f}ms "
              f"{lp['total_ms']:>7.1f}ms")
print(f"{'TOTAL':<30s} {'':>6s} {'':>6s} {'':>8s} {'':>8s} {'':>8s} {'':>8s} "
      f"{prof['total_ms']:>7.1f}ms")


Profile: 55mhz (55 MHz), mode=legacy MMIO
Test image: img_0000.hex, label=7

=== Single Image LeNet-5 (verify=True, profile=True) ===
[verify] run_id = no-git_20221022_011907
  Layer 0 (Conv 1ch 5x5->6ch): 28 packed MVMs (k_pack=21), 66304 cycles
  Conv: feat[1,28,28] * weight[6,1,5,5]  stride=1 pad=0  mode=explicit
  Output spatial: 24×24 = 576 pixels, col_len=25, MVMs=576
  [MATCH]   layer_0 (conv_1x5x5_to_6)  3456 elements
  Layer 1 (Pool 2x2): -> (6, 12, 12)
  Layer 2 (Conv 6ch 5x5->16ch): 13 packed MVMs (k_pack=5), 24700 cycles
  Conv: feat[6,12,12] * weight[16,6,5,5]  stride=1 pad=0  mode=explicit
  Output spatial: 8×8 = 64 pixels, col_len=150, MVMs=64
  [MATCH]   layer_2 (conv_6x5x5_to_16)  1024 elements
  Layer 3 (Pool 2x2): -> (16, 4, 4)
  Layer 4 (FC 256->120): 1552 cycles
  [MATCH]   layer_4 (fc_256x120)  120 elements
  Layer 5 (FC 120->84): 876 cycles
  [MATCH]   layer_5 (fc_120x84)  84 elements
  Layer 6 (FC 84->10): 134 cycles
  [MATCH]   layer_6 (fc_84x10)  10 elements



In [6]:
# ====================================================================
# 6. 批量推理：20 张图，统计正确率 + 平均耗时
# ====================================================================
img_files = sorted(glob.glob(f"{img_dir}/img_????.hex"))
n_images = len(img_files)
print(f"=== Batch Test: {n_images} images ({PROFILE_NAME}, {FABRIC_MHZ} MHz) ===\n")

correct = 0
t0 = time.time()

for img_path in img_files:
    name = os.path.basename(img_path).replace(".hex", "")
    img = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(f"{img_dir}/{name}_label.txt").read().strip())

    pred, logits = model.predict(img.reshape(1, 28, 28))

    if pred == label:
        correct += 1
    else:
        print(f"  {name}: pred={pred} label={label} WRONG")

elapsed = time.time() - t0
print(f"\nProfile : {PROFILE_NAME}")
print(f"Bitstream: {BITSTREAM_NAME}")
print(f"Results : {correct}/{n_images} correct ({100 * correct / n_images:.1f}%)")
print(f"Wall time: {elapsed:.2f}s ({elapsed / n_images * 1000:.1f} ms/image)")


=== Batch Test: 200 images (55mhz, 55 MHz) ===

  img_0018: pred=8 label=3 WRONG

Profile : 55mhz
Bitstream: cim_soc_55mhz.bit
Results : 199/200 correct (99.5%)
Wall time: 338.26s (1691.3 ms/image)
